# 05 · B3: per què les reparacions de bateria acaben sense arreglar

1. Com acaben les reparacions de bateria, comparades amb les de pantalla.
2. Per què: llegim els textos. Una **precodificació automàtica** per paraules clau proposa un motiu per a cada text; després **dues persones** l'han de revisar a `dades/processades/codificacio_bateria.csv` (columnes `codi_1` i `codi_2`). Fins aleshores, el resum és provisional.

In [1]:
import sys
sys.path.insert(0, "..")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import comu

rng = np.random.default_rng(42)

dades = comu.carrega_classificats()
estat = {"Fixed": "Reparat", "Repairable": "Reparable", "End of life": "Final de vida"}
dades["resultat"] = dades["repair_status"].map(estat)

def resultats(df):
    t = pd.crosstab(df["grup_avaria"], df["resultat"]).loc[["Bateria i alimentació", "Pantalla"]]
    return t.join((t.div(t.sum(axis=1), axis=0) * 100).round(1).add_suffix(" %"))

print("Tots els registres:")
display(resultats(dades))
print("Només etiquetes humanes (MobiFix):")
resultats(dades[dades["font_avaria"] == "MobiFix"])

Tots els registres:


resultat,Final de vida,Reparable,Reparat,Final de vida %,Reparable %,Reparat %
grup_avaria,,,,,,
Bateria i alimentació,89,203,210,17.7,40.4,41.8
Pantalla,87,364,514,9.0,37.7,53.3


Només etiquetes humanes (MobiFix):


resultat,Final de vida,Reparable,Reparat,Final de vida %,Reparable %,Reparat %
grup_avaria,,,,,,
Bateria i alimentació,26,96,78,13.0,48.0,39.0
Pantalla,29,232,323,5.0,39.7,55.3


## Precodificació

Llibre de codis (s'aplica el primer que coincideix, en aquest ordre):

| Codi | Què vol dir |
|---|---|
| `inflada` | Bateria inflada, vessada o problema de seguretat |
| `no_era_bateria` | L'avaria era la placa, el port o un altre component |
| `no_sobre` | No es pot obrir, la bateria va enganxada o no es pot canviar, o calen eines especials |
| `peca` | No hi ha la peça, s'ha de demanar o és massa cara |
| `a_casa` | La persona compra la peça i la canvia ella mateixa |
| `no_val_pena` | El mòbil és massa vell o no surt a compte (cost i benefici) |
| `no_consta` | El text no diu per què |

In [2]:
CODIS = {
    "inflada": comu.SIMPTOMES["inflada"] + r"|ausgelaufen|leaked|lekt|lekkage",
    "no_era_bateria": r"motherboard|mainboard|logic ?board|\bboard\b|platine|moederbord|carte mère|placa base|\bport\b|buchse|charging ic"
                      r"|ruhestrom|gleiches verhalten|same (?:problem|behaviou?r) with",
    "no_sobre": r"glued|geklebt|verklebt|lijm|can.?t open|cannot open|could not open|nicht öffnen|niet open|ouvrir|no se puede abrir|åbne"
                r"|inchangeable|non.?remov|not removable|nicht wechselbar|verbaut|special(?:ist)? tools|spezialwerkzeug",
    "peca": r"spare|\bparts?\b|ersatz|onderdeel|pièce|repuesto|reservedel|bestell|order|lieferbar|leverbaar|expensive|teuer|duur|cher|commander|commande",
    "a_casa": r"at home|zu hause|zuhause|thuis|à la maison|en casa|selbst|zelf|herself|himself|themselves|will replace",
    "no_val_pena": r"not worth|lohnt|loont niet|rentable|no merece|too old|zu alt|te oud|trop vieux|unwirtschaftlich"
                   r"|kosten|baten|\bcosts?\b|wirtschaftlich|economic",
}

def codifica(text):
    t = str(text).lower()
    trobats = [c for c, patro in CODIS.items() if pd.Series([t]).str.contains(patro, regex=True).iloc[0]]
    return (trobats[0] if trobats else "no_consta"), ";".join(trobats)

bat = dades[(dades["grup_avaria"] == "Bateria i alimentació") & dades["problem"].notna()]
a_codificar = pd.concat([bat[bat["repair_status"] == "End of life"],
                         bat[bat["repair_status"] == "Repairable"].sample(100, random_state=42)])
a_codificar[["codi_auto", "codis_auto_tots"]] = a_codificar["problem"].apply(lambda t: pd.Series(codifica(t)))
print(f"Textos a codificar: {len(a_codificar)} ({(a_codificar['repair_status'] == 'End of life').sum()} a final de vida)")

Textos a codificar: 189 (89 a final de vida)


In [3]:
fitxer = comu.PROCESSADES / "codificacio_bateria.csv"
if not fitxer.exists():
    (a_codificar[["id", "repair_status", "font_avaria", "problem", "codi_auto", "codis_auto_tots"]]
     .sample(frac=1, random_state=42)
     .assign(codi_1="", codi_2="")
     .to_csv(fitxer, index=False))
    print(f"Generat {fitxer.name}")
print("Codis possibles:", list(CODIS) + ["no_consta"])

Generat codificacio_bateria.csv
Codis possibles: ['inflada', 'no_era_bateria', 'no_sobre', 'peca', 'a_casa', 'no_val_pena', 'no_consta']


## Resum provisional (precodificació automàtica)

In [4]:
t = pd.crosstab(a_codificar["codi_auto"], a_codificar["repair_status"].map(estat))
t.join((t / t.sum() * 100).round(0).add_suffix(" %"))

repair_status,Final de vida,Reparable,Final de vida %,Reparable %
codi_auto,,,,
a_casa,0,2,0.0,2.0
inflada,6,4,7.0,4.0
no_consta,67,78,75.0,78.0
no_era_bateria,7,3,8.0,3.0
no_sobre,2,0,2.0,0.0
no_val_pena,4,0,4.0,0.0
peca,3,13,3.0,13.0


In [5]:
with pd.option_context("display.max_colwidth", 140):
    display(a_codificar[a_codificar["repair_status"] == "End of life"][["codi_auto", "problem"]].head(15))

,codi_auto,problem
169,inflada,"Bitte Akku tauschen. - Alter Akku ist im Gerät ausgelaufen und hat starke, korrosive Beschädigungen angerichtet. Nicht reparabel. [UE] {..."
240,no_consta,der Akku entlädt sich zu schnell.
520,peca,"Akku zu schwach, Ersatzakku vorhanden."
531,no_consta,Akku lädt nicht.
536,no_era_bateria,Akku und Buchsen defekt.
571,no_consta,Akku defekt.
665,no_era_bateria,Auf Grund sehr kurzer Akkulaufzeit hat der Besitzer einen neuen Akku gekauft - gleiches Verhalten mit dem neuen Akku. Ruhestromaufnahme ...
718,no_consta,Akkutausch.
765,inflada,Akku aufgebläht.
778,no_consta,Smartphone -Akku lädt nicht -.


## Validació per dues persones

In [6]:
from sklearn.metrics import cohen_kappa_score

codificat = pd.read_csv(fitxer).dropna(subset=["codi_1", "codi_2"])
if len(codificat) < 50:
    print(f"Codificació manual pendent: {len(codificat)} textos codificats per totes dues persones")
else:
    print(f"Kappa entre persones: {cohen_kappa_score(codificat['codi_1'], codificat['codi_2']):.2f}")
    acord = codificat[codificat["codi_1"] == codificat["codi_2"]]
    print(f"Encert de la precodificació on coincideixen: {(acord['codi_auto'] == acord['codi_1']).mean():.0%}")
    display(pd.crosstab(acord["codi_1"], acord["repair_status"]))

Codificació manual pendent: 0 textos codificats per totes dues persones


## Motius registrats pels voluntaris

El camp `repair_barrier_if_end_of_life` només està omplert en una part dels finals de vida.

In [7]:
eol = bat[bat["repair_status"] == "End of life"]
print(f"Amb motiu: {eol['repair_barrier_if_end_of_life'].notna().sum()} de {len(eol)}")
eol["repair_barrier_if_end_of_life"].value_counts()

Amb motiu: 32 de 89


repair_barrier_if_end_of_life
Spare parts too expensive           9
No way to open product              6
Spare parts not available           6
Lack of equipment                   5
Repair information not available    4
Product too worn out                2
Name: count, dtype: int64